# 工商业用户电费回收风险模型的多次划分稳健性评价 · 教学参考

这是帮助理解写法的参考实现，不是官方标准答案。请在 `answer.ipynb` 独立作答；不要在本文件中覆盖自己的草稿。

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

data = pd.read_csv("04/SS-5-5-5-01_payment_risk.csv")
data = data.dropna().drop_duplicates().reset_index(drop=True)
X = data[["月均用电量", "用电波动率", "平均账期", "历史违约次数", "客户星级", "行业类型", "合同容量", "入网年限"]]
y = data["是否违约"]
results = []
for seed in [2026, 42, 100]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    model = RandomForestClassifier(n_estimators=100, random_state=seed)
    model.fit(X_train_s, y_train)
    pred = model.predict(X_test_s)
    results.append((accuracy_score(y_test, pred), f1_score(y_test, pred)))
print(results)
acc_gap = max(item[0] for item in results) - min(item[0] for item in results)
f1_gap = max(item[1] for item in results) - min(item[1] for item in results)
print("结果基本稳定" if acc_gap <= 0.10 and f1_gap <= 0.10 else "结果波动较大")
